In [ ]:
from sentence_transformers import SentenceTransformer

# Load the BGE-M3 model from HuggingFace on CPU
model = SentenceTransformer('BAAI/bge-m3', device='cpu')

# Input sentences for inference
sentences = [
    "BGE-M3 is a multi-lingual, multi-functional, and multi-granularity retrieval model.",
    "This is an example of running inference on CPU."
]

# Generate dense embeddings
embeddings = model.encode(sentences, normalize_embeddings=True)

print(f"Embedding Shape: {embeddings.shape}")


In [ ]:
# Input sentences for inference
sentences = [
    "BGE-M3 is a multi-lingual, multi-functional, and multi-granularity retrieval model.",
    "This is an example of running inference on CPU."
]

# Generate dense embeddings
embeddings = model.encode(sentences, normalize_embeddings=True)

print(f"Embedding Shape: {embeddings.shape}")

In [ ]:
import pandas as pd
from sentence_transformers import util

def get_sentence_similarity_matrix(sentences, model):
    """
    Computes a cosine similarity matrix for a list of sentences using SBERT
    and returns it as a pandas DataFrame for improved readability.
    """
    embeddings = model.encode(sentences)
    similarity_matrix = util.cos_sim(embeddings, embeddings).numpy()
    return pd.DataFrame(similarity_matrix, index=sentences, columns=sentences)


In [ ]:
my_sentences = [
    "I am a bit late on my homework",
    "Homework takes a lot of time",
    "The weather is so nice today",
    "Climate change is a huge issue that all of us should pay attention to"
]

In [ ]:
my_sentences = [
    "I am a bit late on my homework",
    "Homework takes a lot of time",
    "The weather is so nice today"
]

In [ ]:
get_sentence_similarity_matrix(
    my_sentences,
    model
)

In [ ]:
get_sentence_similarity_matrix(
    my_sentences,
    model
)

In [ ]:
import duckdb

# Connect to DuckDB (persistent or in-memory)
con = duckdb.connect('vector_storage.db')

# Install and load the vector similarity search (vss) extension
con.execute("INSTALL vss; LOAD vss;")

def store_embeddings(table_name, ids, embeddings):
    """
    Stores vector embeddings in DuckDB using the vss extension.
    :param table_name: Name of the table to create/insert into
    :param ids: List of identifiers
    :param embeddings: List of list/arrays (all must have the same dimension)
    """
    dimension = len(embeddings[0])
    
    # Create table with a fixed-size array column for embeddings
    con.execute(f"CREATE TABLE IF NOT EXISTS {table_name} (id INTEGER, vec FLOAT[{dimension}])")
    
    # Batch insert the embeddings
    data = list(zip(ids, embeddings))
    con.executemany(f"INSERT INTO {table_name} VALUES (?, ?)", data)
    
    # Create an HNSW index for efficient vector similarity search
    con.execute(f"CREATE INDEX IF NOT EXISTS {table_name}_idx ON {table_name} USING HNSW (vec)")

# Example usage:
# ids = [1, 2, 3]
# embeddings = [[0.1, 0.2, 0.3], [0.4, 0.5, 0.6], [0.7, 0.8, 0.9]]
# store_embeddings("my_vectors", ids, embeddings)

